### Imports

In [1]:
import pandas as pd
import numpy as np
import requests
from io import StringIO
from time import sleep

### Load county FIPS list
Uses Census file to map county name to  its respective FIPS code. Census county mapping files by year downloaded from https://www.census.gov/geographies/reference-files/time-series/geo/gazetteer-files.html.

In [20]:
counties = pd.read_csv(
    "raw-files/2016_counties_national.txt",
    sep="\t",
    dtype=str,
    encoding="latin1"
)

In [21]:
counties.head()
counties.columns

Index(['USPS', 'GEOID', 'ANSICODE', 'NAME', 'ALAND', 'AWATER', 'ALAND_SQMI',
       'AWATER_SQMI', 'INTPTLAT',
       'INTPTLONG                                                                                                               '],
      dtype='str')

In [22]:
counties["county_fips"] = counties["GEOID"].str.zfill(5)

In [23]:
print(counties[["USPS","NAME","county_fips"]].head())
print(len(counties))

  USPS            NAME county_fips
0   AL  Autauga County       01001
1   AL  Baldwin County       01003
2   AL  Barbour County       01005
3   AL     Bibb County       01007
4   AL   Blount County       01009
3220


In [24]:
exclude_state_fips = {"02", "15", "60", "66", "69", "72", "78"}

counties_conus = counties.loc[
    ~counties["county_fips"].str[:2].isin(exclude_state_fips)
].copy()

county_fips_list = counties_conus["county_fips"].tolist()

len(county_fips_list)

3108

In [25]:
# test one county first
test_url = "https://nassgeodata.gmu.edu/axis2/services/CDLService/GetCDLStat"

test_params = {
    "year": 2016,
    "fips": "01001",   # Autauga County, AL
    "format": "csv"
}

r = requests.get(test_url, params=test_params, timeout=60)
print(r.status_code)
print(r.text[:500])

200
<ns1:GetCDLStatResponse xmlns:ns1="http://cropscape.csiss.gmu.edu/CDLService/"><returnURL>https://nassgeodata.gmu.edu/webservice/nass_data_cache/byfips/CDL_2016_01001.csv</returnURL></ns1:GetCDLStatResponse>


In [26]:
test_df = pd.read_csv(StringIO(r.text))
test_df.head()

,"<ns1:GetCDLStatResponse xmlns:ns1=""http://cropscape.csiss.gmu.edu/CDLService/""><returnURL>https://nassgeodata.gmu.edu/webservice/nass_data_cache/byfips/CDL_2016_01001.csv</returnURL></ns1:GetCDLStatResponse>"


### Helper functions for CropScape

In [18]:
BASE_URL = "https://nassgeodata.gmu.edu/axis2/services/CDLService/GetCDLStat"

def chunk_list(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i + size]

def fetch_cdl_stats(year, fips_batch, timeout=120):
    params = {
        "year": year,
        "fips": ",".join(fips_batch),
        "format": "csv"
    }
    
    r = requests.get(BASE_URL, params=params, timeout=timeout)
    r.raise_for_status()
    
    return pd.read_csv(StringIO(r.text))

### Trying one year first (2016)

In [19]:
year = 2016
frames = []
failed_batches = []

for i, batch in enumerate(chunk_list(county_fips_list, 100), start=1):
    try:
        df = fetch_cdl_stats(year, batch)
        df["year"] = year
        frames.append(df)
        print(f"Batch {i} succeeded ({len(batch)} counties)")
    except Exception as e:
        failed_batches.append({
            "batch_number": i,
            "first_fips_in_batch": batch[0],
            "error": str(e)
        })
        print(f"Batch {i} FAILED ({batch[0]}): {e}")
    
    sleep(0.5)

cdl_raw_2016 = pd.concat(frames, ignore_index=True)

print(cdl_raw_2016.shape)
cdl_raw_2016.head()

Batch 1 FAILED (01001): 500 Server Error: Internal Server Error for url: https://nassgeodata.gmu.edu/axis2/services/CDLService/GetCDLStat?year=2016&fips=01001%2C01003%2C01005%2C01007%2C01009%2C01011%2C01013%2C01015%2C01017%2C01019%2C01021%2C01023%2C01025%2C01027%2C01029%2C01031%2C01033%2C01035%2C01037%2C01039%2C01041%2C01043%2C01045%2C01047%2C01049%2C01051%2C01053%2C01055%2C01057%2C01059%2C01061%2C01063%2C01065%2C01067%2C01069%2C01071%2C01073%2C01075%2C01077%2C01079%2C01081%2C01083%2C01085%2C01087%2C01089%2C01091%2C01093%2C01095%2C01097%2C01099%2C01101%2C01103%2C01105%2C01107%2C01109%2C01111%2C01113%2C01115%2C01117%2C01119%2C01121%2C01123%2C01125%2C01127%2C01129%2C01131%2C01133%2C02013%2C02016%2C02020%2C02050%2C02060%2C02068%2C02070%2C02090%2C02100%2C02105%2C02110%2C02122%2C02130%2C02150%2C02158%2C02164%2C02170%2C02180%2C02185%2C02188%2C02195%2C02198%2C02220%2C02230%2C02240%2C02261%2C02275%2C02282%2C02290%2C04001%2C04003%2C04005%2C04007&format=csv
Batch 2 FAILED (04009): 500 Server Err

KeyboardInterrupt: 

### Inspect columns

In [ ]:
print(cdl_raw_2016.columns.tolist())
cdl_raw_2016.head(10)

### Save raw 2016 file

In [ ]:
cdl_raw_2016.to_csv("raw-files/cdl_raw_2016.csv", index=False)

failed_2016 = pd.DataFrame(failed_batches)
failed_2016.to_csv("raw-files/cdl_failed_batches_2016.csv", index=False)

failed_2016.head()

### Standardize column names

In [ ]:
cdl_2016 = cdl_raw_2016.rename(columns={
    "FIPS": "county_fips",
    "fips": "county_fips",
    "County": "county_name",
    "county": "county_name",
    "Category": "class_name",
    "category": "class_name",
    "Code": "class_code",
    "code": "class_code",
    "Acres": "acres",
    "acres": "acres",
    "Percent": "percent",
    "percent": "percent",
    "Pixels": "pixel_count",
    "Pixel": "pixel_count",
    "pixels": "pixel_count"
}).copy()

cdl_2016["county_fips"] = cdl_2016["county_fips"].astype(str).str.zfill(5)
cdl_2016["acres"] = pd.to_numeric(cdl_2016["acres"], errors="coerce")
cdl_2016["percent"] = pd.to_numeric(cdl_2016["percent"], errors="coerce")

cdl_2016.head()

### Merge county names/state abbreviations back on if helpful

In [ ]:
cdl_2016 = cdl_raw_2016.rename(columns={
    "FIPS": "county_fips",
    "fips": "county_fips",
    "County": "county_name",
    "county": "county_name",
    "Category": "class_name",
    "category": "class_name",
    "Code": "class_code",
    "code": "class_code",
    "Acres": "acres",
    "acres": "acres",
    "Percent": "percent",
    "percent": "percent",
    "Pixels": "pixel_count",
    "Pixel": "pixel_count",
    "pixels": "pixel_count"
}).copy()

cdl_2016["county_fips"] = cdl_2016["county_fips"].astype(str).str.zfill(5)
cdl_2016["acres"] = pd.to_numeric(cdl_2016["acres"], errors="coerce")
cdl_2016["percent"] = pd.to_numeric(cdl_2016["percent"], errors="coerce")

cdl_2016.head()